# Routing in Express.js

Routing determines how your application responds to client requests at specific endpoints. Each endpoint is defined by a unique URL path and a specific HTTP request method like `GET`, `POST`, `PUT`, or `DELETE`.

## Basic Routing Syntax

Routes are mapped using the app instance with this structure:

```javascript
app.METHOD(PATH, HANDLER);
```

| Part | Meaning |
| --- | --- |
| `app` | The Express application instance |
| `METHOD` | The lowercase HTTP method (`get`, `post`, `put`, `patch`, `delete`) |
| `PATH` | The URL path on the server |
| `HANDLER` | Callback executed when the route matches, receiving `req` and `res` |

### Core CRUD Example

```javascript
const express = require('express');
const app = express();

// Parse JSON bodies (required for POST/PUT)
app.use(express.json());

// GET: Read data
app.get('/api/users', (req, res) => {
    res.json({ message: "Fetch all users" });
});

// POST: Create data
app.post('/api/users', (req, res) => {
    res.json({ message: "User created", data: req.body });
});

// PUT: Update data
app.put('/api/users', (req, res) => {
    res.json({ message: "User updated" });
});

// DELETE: Delete data
app.delete('/api/users', (req, res) => {
    res.json({ message: "User deleted" });
});

app.listen(3000, () => console.log('Server running on port 3000'));
```

Note that in a real API, `PUT` and `DELETE` target a *specific* resource, so those paths would be `/api/users/:id` — you can't update or delete "all users" as a collection. See the RESTful conventions table below.

## Extracting Dynamic Data

Express extracts dynamic data from incoming requests via route parameters or query strings.

| Feature | Scope | Example URL | Definition Path | Accessing the Data |
| --- | --- | --- | --- | --- |
| **Route parameters** | Required identifiers embedded in the path | `/users/42` | `/users/:id` | `req.params.id` |
| **Query parameters** | Optional key-value pairs after a `?` | `/search?name=alice` | `/search` | `req.query.name` |

```javascript
// Handling route parameters
app.get('/users/:id', (req, res) => {
    const userId = req.params.id;
    res.send(`Viewing profile for user ${userId}`);
});

// Handling query parameters
app.get('/search', (req, res) => {
    const searchTerm = req.query.name;
    res.send(`Searching database for: ${searchTerm}`);
});
```

A path can hold several parameters — `/users/:userId/posts/:postId` gives you both in `req.params`. Everything extracted this way is a **string**, so convert before doing arithmetic or strict comparisons.

## Route Order Matters

Express checks routes **top to bottom** and uses the first match. This makes ordering a real source of bugs:

```javascript
// ❌ WRONG — /users/new never runs
app.get('/users/:id', handler);   // matches "new" as an id
app.get('/users/new', handler);

// ✅ RIGHT — specific paths before dynamic ones
app.get('/users/new', handler);
app.get('/users/:id', handler);
```

The general rule: **static segments before parameterised ones.**

## Handling Unmatched Routes (404)

If nothing matches, Express sends its own bare HTML error page. Add a catch-all *after* all other routes to control that:

```javascript
// Registered last — only reached when nothing above matched
app.use((req, res) => {
    res.status(404).json({ error: 'Route not found' });
});
```

## RESTful Conventions

Routing is a technical mechanism; REST is the convention most APIs follow when using it.

| Method | Path | Purpose | Typical status |
| --- | --- | --- | --- |
| `GET` | `/api/users` | List all users | `200` |
| `GET` | `/api/users/:id` | Fetch one user | `200` / `404` |
| `POST` | `/api/users` | Create a user | `201` |
| `PUT` | `/api/users/:id` | Replace a user entirely | `200` |
| `PATCH` | `/api/users/:id` | Update selected fields | `200` |
| `DELETE` | `/api/users/:id` | Remove a user | `204` |

Two habits worth adopting: use **plural nouns** for collections (`/users`, not `/getUser`), and let the HTTP verb carry the action. `POST /api/deleteUser` is a common beginner smell.

## Modular Routing with `express.Router`

Defining every endpoint inside a single `app.js` quickly becomes unmaintainable. `express.Router` is a "mini-application" that groups related routes into dedicated files.

### 1. Define routes in a separate file (`routes/users.js`)

```javascript
const express = require('express');
const router = express.Router(); // Create isolated router instance

// Paths are relative to where this router is mounted
router.get('/', (req, res) => {
    res.send('User directory list');
});

router.get('/:id', (req, res) => {
    res.send(`Details for user ${req.params.id}`);
});

module.exports = router; // Export the router object
```

### 2. Import and mount it in your entry file (`app.js`)

```javascript
const express = require('express');
const app = express();
const userRouter = require('./routes/users'); // Import modular routes

// Mount the router under the base path "/users"
app.use('/users', userRouter);

app.listen(3000);
```

With this architecture, the exposed endpoints become `/users` and `/users/:id`.

A router can also carry its own middleware, which applies only to routes in that file:

```javascript
router.use(requireAuth);   // protects every route in this router only
```

By default a child router does **not** see the parent's params. If `routes/posts.js` is mounted at `/users/:userId/posts` and you need `req.params.userId` inside it, create the router with `express.Router({ mergeParams: true })`.

## Advanced Routing Utilities

### `app.route()`

Avoids repeating the same path across multiple HTTP verbs by chaining handlers:

```javascript
app.route('/api/books')
   .get((req, res) => res.send('Get all books'))
   .post((req, res) => res.send('Add a new book'));
```

### `app.all()`

A catch-all that applies to a path across every HTTP method — typically used for cross-cutting concerns like auth logging:

```javascript
app.all('/api/*', (req, res, next) => {
    console.log('API endpoint accessed. Checking auth...');
    next(); // Pass control to the specific matching route handler
});
```

> **Express 5 change:** bare `*` wildcards are no longer valid. Use a named wildcard — `'/api/*splat'` — or `'/api/{*splat}'` to also match `/api` itself. Code written for Express 4 will throw a path-to-regexp error on Express 5.

### Multiple Handlers on One Route

Any route accepts a chain of functions, which is how per-route middleware works:

```javascript
app.post('/api/users', validateBody, requireAuth, (req, res) => {
    res.status(201).json({ created: true });
});
```

Each function either calls `next()` to continue the chain or sends a response to end it.

## Quick Reference

| Tool | Use it for |
| --- | --- |
| `app.get/post/put/delete()` | A single route on a single method |
| `app.route(path)` | Several methods sharing one path |
| `app.all(path)` | Every method on one path (middleware-style) |
| `express.Router()` | Grouping related routes into their own file |
| `app.use(mw)` | Middleware applied to everything below it |